# PBL6: đợt 3 — thêm seed và tăng cường dữ liệu nền màu

**Mục đích.** (1) Đo nhiễu giữa các seed để biết chênh lệch nào giữa M4 và M5 là thật; (2) thử dữ liệu nhuộm màu giấy mạnh hơn để sửa lỗi model tẩy trắng trang nền màu.

| Run | Cấu hình | Seed |
|---|---|---|
| `m4_s1`, `m4_s2` | như M4 chạy lại (không cải tiến) | 1, 2 |
| `m5_all_s1`, `m5_all_s2` | `--edge-w 1.0 --paper-color 0.2 --identity-p 0.1` | 1, 2 |
| `m5d_color50` | như M5 nhưng `--paper-color 0.5` | 0 |
| `m5e_color50_noedge` | `--paper-color 0.5 --identity-p 0.1` (không loss gradient) | 0 |

Seed 0 của `m4r_baseline` và `m5_all` lấy từ đợt 2 (`checkpoints/retrain2`), nên mỗi cấu hình có đủ 3 seed.

**Cần có trên Drive:** như đợt 2, cộng `checkpoints/retrain2/` (checkpoint đợt 2) và `code/pbl6_pipeline_code.zip` bản mới.
Chạy từ trên xuống; Runtime → A100/H100.


In [ ]:
import json, os, subprocess, sys, glob, shutil, time
from pathlib import Path
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
NPROC = os.cpu_count(); print('vCPU:', NPROC)

from google.colab import drive
drive.mount('/content/drive')
D = Path('/content/drive/MyDrive/PBL6')
CK_OLD = D / 'checkpoints' / 'retrain'       # đợt 1 (M4...), chỉ đọc
CK = D / 'checkpoints' / 'retrain3'          # đợt 3: mỗi run một thư mục con
CK2 = D / 'checkpoints' / 'retrain2'         # đợt 2 (m4r_baseline, m5_all seed 0), chỉ đọc
RES = D / 'results' / 'retrain3'
FIG = RES / 'figures'
for p in (CK, RES, FIG): p.mkdir(parents=True, exist_ok=True)
C = Path('/content/pbl6_pipeline')
subprocess.run(['unzip', '-oq', str(D / 'code' / 'pbl6_pipeline_code.zip'), '-d', '/content'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lpips'], check=True)
print(sorted(p.name for p in C.glob('*.py')))


## 1. Dữ liệu

ESP32: giải nén ba zip của bước N2 vào `/content/esp32`.


In [ ]:
E = Path('/content/esp32'); E.mkdir(exist_ok=True)
for z in ['n2_patches.zip', 'n2_pages_layout.zip', 'n2_ocr_lines.zip']:
    subprocess.run(['unzip', '-oq', str(D / 'data' / z), '-d', str(E)], check=True)

def lift(name):
    # zip có thể chứa thêm một thư mục bọc ngoài (vd n2_data/patches): đưa về E/<name>
    if (E / name).exists():
        return
    hits = [p for p in E.rglob(name) if p.parent != E]
    if hits:
        shutil.move(str(hits[0]), str(E / name))
for n in ['patches', 'pages', 'crops', 'labels', 'lines.jsonl']:
    lift(n)
for n in ['patches/train/esp', 'patches/val/esp', 'pages/esp_raw', 'pages/scan', 'lines.jsonl']:
    print(f'{n:22s}', 'OK' if (E / n).exists() else 'THIẾU')


RealDAE: giải nén rồi chuẩn hóa về `/content/realdae/{train,val,test}/{inp,gt}` (cạnh dài 1024 px; 50 ảnh train tách làm val).


In [ ]:
R_RAW, R = Path('/content/realdae_raw'), Path('/content/realdae')
HAVE_REALDAE = (D / 'data' / 'realdae_raw.zip').exists()
if HAVE_REALDAE:
    subprocess.run(['unzip', '-oq', str(D / 'data' / 'realdae_raw.zip'), '-d', str(R_RAW)], check=True)
    subprocess.run([sys.executable, C / 'prep_realdae.py', '--src', R_RAW, '--tree'])
    subprocess.run([sys.executable, C / 'prep_realdae.py', '--src', R_RAW, '--out', R,
                    '--long-side', '1024', '--val', '50', '--seed', '0'], check=True)
    print(json.loads((R / 'manifest.json').read_text())['counts'])
else:
    print('Chưa có realdae_raw.zip trên Drive → chỉ dùng ESP32 + tổng hợp')


Tập test ngoài (không dùng khi train): **DocAligner** (130 ảnh chụp điện thoại, ảnh chuẩn là bản scan) và **Jung** (87 ảnh bóng đổ).
Tập test tổng hợp cố định (6 loại + mix, 3 mức, seed 123) từ ảnh chuẩn RealDAE test và 10 trang scan ESP32 test.


In [ ]:
EXT = Path('/content/ext'); EXT.mkdir(exist_ok=True)
for z in ['eval_docaligner.zip', 'eval_jung.zip']:
    subprocess.run(['unzip', '-oq', str(D / 'data' / z), '-d', str(EXT)], check=True)

TEST_PAGES = [3, 4, 7, 9, 11, 14, 25, 32, 38, 44]
EC = Path('/content/esp32_test_clean'); EC.mkdir(exist_ok=True)
for p in TEST_PAGES:
    shutil.copy(E / 'pages' / 'scan' / f'{p:06d}.png', EC / f'esp{p:02d}.png')
clean = [str(EC)] + ([str(R / 'test' / 'gt')] if HAVE_REALDAE else [])
SYN = Path('/content/synth_test')
subprocess.run([sys.executable, C / 'degrade.py', '--clean', *clean, '--out', SYN,
                '--long-side', '1024', '--max-images', '60', '--seed', '123'], check=True)

# tập "ảnh sạch → ảnh sạch" để đo model có làm hỏng ảnh đã tốt không (phép thử cho --identity-p)
CLEAN = Path('/content/clean_identity'); (CLEAN / 'inp').mkdir(parents=True, exist_ok=True); (CLEAN / 'gt').mkdir(exist_ok=True)
src = sorted((R / 'test' / 'gt').glob('*.png'))[:60] if HAVE_REALDAE else sorted(EC.glob('*.png'))
for f in src:
    shutil.copy(f, CLEAN / 'inp' / f.name); shutil.copy(f, CLEAN / 'gt' / f.name)

import pandas as pd
def count(d): return len(list(Path(d).glob('*.png'))) + len(list(Path(d).glob('*.jpg')))
def trio(*v): return ' / '.join(str(x) for x in v)
tab = pd.DataFrame([
    ['ESP32 patch (train / val)', trio(len(list((E/'patches/train/esp').glob('*.png'))), len(list((E/'patches/val/esp').glob('*.png'))))],
    ['RealDAE (train / val / test)', trio(*(count(R/s/'gt') for s in ('train', 'val', 'test'))) if HAVE_REALDAE else 'không có'],
    ['Tổng hợp test (ảnh)', len(list(SYN.glob('*/L*/inp/*.png')))],
    ['DocAligner test', count(EXT/'docaligner/gt')],
    ['Jung test', count(EXT/'jung/gt')],
], columns=['Tập', 'Số lượng'])
tab


**Ví dụ dữ liệu huấn luyện tổng hợp.** Một ảnh chuẩn, các loại suy biến (cường độ tăng dần) và ảnh nhuộm màu giấy (cải tiến `--paper-color`).


In [ ]:
sys.path.insert(0, str(C))
import numpy as np, cv2, matplotlib.pyplot as plt
from degrade import degrade, level_config
from train_restore import tint_paper

def rd(p): return cv2.imread(str(p))[..., ::-1]
ref = rd(sorted((R / 'test' / 'gt' if HAVE_REALDAE else EC).glob('*.png'))[3])
ref = cv2.resize(ref, None, fx=640 / max(ref.shape[:2]), fy=640 / max(ref.shape[:2]), interpolation=cv2.INTER_AREA)
rng = np.random.default_rng(1)
types = ['shading', 'color', 'lowlight', 'blur', 'noise', 'jpeg']
fig, ax = plt.subplots(3, 4, figsize=(16, 11))
panels = [('Ảnh chuẩn', ref)] + [(f'{t} mức 3', degrade(ref, level_config(t, 3, rng), rng)) for t in types] +          [('mix mức 2', degrade(ref, level_config('mix', 2, rng), rng))] +          [(f'nhuộm giấy #{i}', tint_paper(ref, rng)) for i in range(4)]
for a, (t, im) in zip(ax.ravel(), panels):
    a.imshow(im); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.savefig(FIG / 'du_lieu_tong_hop.png', dpi=110); plt.show()


## 3. Train (tối đa 3 run song song, 6 run, khoảng 30–40 phút trên A100)


In [ ]:
COMMON = ['--iters', '8000', '--bs', '16', '--crop', '192', '--eval-every', '500', '--seed', '0']
BASE = ['--data', *DATA_ALL, '--synth', '0.5']
RUNS = {
    'm4_s1':              BASE + ['--seed', '1'],
    'm4_s2':              BASE + ['--seed', '2'],
    'm5_all_s1':          BASE + ['--edge-w', '1.0', '--paper-color', '0.2', '--identity-p', '0.1', '--seed', '1'],
    'm5_all_s2':          BASE + ['--edge-w', '1.0', '--paper-color', '0.2', '--identity-p', '0.1', '--seed', '2'],
    'm5d_color50':        BASE + ['--edge-w', '1.0', '--paper-color', '0.5', '--identity-p', '0.1'],
    'm5e_color50_noedge': BASE + ['--paper-color', '0.5', '--identity-p', '0.1'],
}
WORKERS = max(2, (NPROC - 1) // len(RUNS))
procs = {}
PARALLEL = 3      # 5 run cùng lúc từng hết bộ nhớ GPU trên A100 40 GB (CUDA OOM): chỉ chạy tối đa 3 run một lúc
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
pending = []

def launch(name, extra, resume=False):
    out = CK / name
    out.mkdir(parents=True, exist_ok=True)
    # COMMON trước, extra sau: đối số trùng thì giá trị sau (của run) thắng
    cmd = [sys.executable, str(C / 'train_restore.py'), *COMMON, *extra, '--out', str(out),
           '--num-workers', str(WORKERS)] + (['--resume'] if resume else [])
    log = open(out / 'stdout.log', 'a')
    procs[name] = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    print('chạy', name, 'pid', procs[name].pid)

def tick():
    # khởi chạy các run đang chờ khi số run đang chạy < PARALLEL; ô theo dõi bên dưới gọi hàm này
    running = [n for n, pr in procs.items() if pr.poll() is None]
    while pending and len(running) < PARALLEL:
        n = pending.pop(0)
        launch(n, RUNS[n]); running.append(n)

pending = list(RUNS)
tick()


In [ ]:
# Theo dõi + đường cong train. Chạy lại ô này bất cứ lúc nào.
import matplotlib.pyplot as plt, pandas as pd

def read_log(name):
    f = CK / name / 'log.jsonl'
    if not f.exists() or not f.read_text().strip():
        return pd.DataFrame()
    return pd.DataFrame([json.loads(l) for l in f.read_text().splitlines()])

tick()      # đủ chỗ trống thì chạy tiếp các run đang chờ
if pending: print('đang chờ:', pending)
status = []
for name in RUNS:
    df = read_log(name)
    rc = procs[name].poll() if name in procs else None
    chua = name in pending
    last = df.iloc[-1] if len(df) else {}
    status.append({'run': name, 'trạng thái': 'chờ' if chua else 'đang chạy' if rc is None and name in procs else f'xong (mã {rc})' if name in procs else '?',
                   'vòng': int(last['iter']) if len(df) else 0, 'val PSNR TB': last.get('val_score'),
                   'vòng/giây': last.get('it_per_s'), 'chờ dữ liệu': last.get('data_wait_frac')})
display(pd.DataFrame(status))

logs = {n: read_log(n) for n in RUNS}
fig, ax = plt.subplots(1, 4, figsize=(21, 4.2))
for n, df in logs.items():
    if not len(df): continue
    ax[0].plot(df['iter'], df['train_l1'], label=n)
    ax[1].plot(df['iter'], df['val_score'], label=n)
    sy = [c for c in df.columns if c == 'val_synth_psnr']
    if sy: ax[2].plot(df['iter'], df['val_synth_psnr'], label=n)
    ax[3].plot(df['iter'], df['it_per_s'], label=n)
for a, t in zip(ax, ['Loss L1 train', 'PSNR val trên ảnh thật (TB các nguồn)', 'PSNR val trên ảnh tổng hợp', 'Tốc độ (vòng/giây)']):
    a.set_title(t); a.set_xlabel('vòng'); a.grid(alpha=.3); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG / 'duong_cong_train.png', dpi=110); plt.show()
print(subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used', '--format=csv'], capture_output=True, text=True).stdout)


In [ ]:
# Nếu runtime bị ngắt: chạy lại các ô mục 0-1 (mount, dữ liệu) và ô định nghĩa RUNS/launch (bỏ vòng for cuối), rồi chạy ô này
def target_iters(extra):
    return int(extra[extra.index('--iters') + 1]) if '--iters' in extra else 8000

pending = [n for n, extra in RUNS.items()
           if not len(read_log(n)) or read_log(n)['iter'].iloc[-1] < target_iters(extra)]
RUNS_RESUME = True
def tick():
    running = [n for n, pr in procs.items() if pr.poll() is None]
    while pending and len(running) < PARALLEL:
        n = pending.pop(0)
        launch(n, RUNS[n], resume=True); running.append(n)
tick()


## 4. Đánh giá (cùng giao thức đợt 2: RealDAE test, tổng hợp, DocAligner, Jung, ảnh sạch, ESP32)


In [ ]:
def ev(tag, setdir, *method):
    out = RES / f'{tag}.json'
    subprocess.run([sys.executable, C / 'eval_sets.py', '--set', setdir, *method, '--lpips', '--out', out], check=True)

SETS = {'synth': SYN, 'docaligner': EXT / 'docaligner', 'jung': EXT / 'jung', 'clean_identity': CLEAN}
if HAVE_REALDAE:
    SETS['realdae'] = R / 'test'

CKPTS = {'m4_s0': CK2 / 'm4r_baseline' / 'best.pt', 'm5_all_s0': CK2 / 'm5_all' / 'best.pt'}
CKPTS.update({p.parent.name: p for p in sorted(CK.glob('*/best.pt'))})
CKPTS = {k: v for k, v in CKPTS.items() if v.exists()}
print('Sẽ chấm:', list(CKPTS))

for sname, sdir in SETS.items():
    ev(f'm0_identity__{sname}', sdir, '--method', 'identity')
    if sname != 'clean_identity':
        ev(f'm1_classic__{sname}', sdir, '--method', 'classic')
    for run, ck in CKPTS.items():
        ev(f'{run}__{sname}', sdir, '--method', 'model', '--ckpt', ck, '--amp', 'bf16')


Trang ESP32 (cả trang):


In [ ]:
for run, ck in CKPTS.items():
    v = 'esp_' + run
    subprocess.run([sys.executable, C / 'restore_eval.py', '--data', E, '--ckpt', ck, '--variant', v, '--splits', 'test'], check=True)
    shutil.copy(E / 'eval' / f'restore_{v}.json', RES / f'{run}__esp32_pages.json')


## 5. Kết quả


In [ ]:
import numpy as np, matplotlib.pyplot as plt
rows = []
for f in sorted(RES.glob('*__*.json')):
    run, sname = f.stem.split('__')
    r = json.loads(f.read_text())
    if 'groups' in r:
        for g, v in r['groups'].items():
            rows.append({'run': run, 'set': sname, 'group': g, **v['mean']})
    else:
        rows.append({'run': run, 'set': sname, 'group': 'test', **r['mean']})
df = pd.DataFrame(rows); df.to_csv(RES / 'summary.csv', index=False)
NAMES = {'realdae': 'RealDAE', 'synth': 'Tổng hợp', 'docaligner': 'DocAligner', 'jung': 'Jung', 'esp32_pages': 'ESP32'}
def overall(sname, metric='psnr'):
    d = df[df['set'] == sname]
    if sname == 'synth': d = d[d['group'] != '.']
    return d.groupby('run')[metric].mean()
main = pd.DataFrame({NAMES[s]: overall(s) for s in NAMES if s in set(df['set'])})
display(main.round(2))
main.round(3).to_csv(RES / 'summary_main_psnr.csv')

# nhóm cấu hình: mean ± std qua seed (std mẫu, n-1)
def cfg(r):
    return 'M4' if r.startswith('m4_s') else 'M5 (cả ba)' if r.startswith('m5_all_s') else r
g = main.drop(index=[r for r in main.index if r in ('m0_identity', 'm1_classic')], errors='ignore').copy()
g['cfg'] = [cfg(r) for r in g.index]
agg = g.groupby('cfg').agg(['mean', 'std', 'count'])
out = pd.DataFrame({c: [f"{agg[(c,'mean')][k]:.2f} ± {agg[(c,'std')][k]:.2f}" if agg[(c,'count')][k] > 1 else f"{agg[(c,'mean')][k]:.2f}" for k in agg.index] for c in main.columns}, index=agg.index)
out['số seed'] = [int(agg[(main.columns[0], 'count')][k]) for k in agg.index]
print('PSNR theo cấu hình, trung bình ± độ lệch chuẩn qua các seed (dB)'); display(out)
out.to_csv(RES / 'summary_seeds.csv')


In [ ]:
# Phép thử ảnh sạch và trang nền màu
ci = overall('clean_identity').to_frame('PSNR ảnh ra so với ảnh sạch (dB)').drop(index='m0_identity', errors='ignore')
print('Ảnh sạch → ảnh sạch (cao hơn là tốt hơn)'); display(ci.round(2))
def per_image(run, sname):
    g = json.loads((RES / f'{run}__{sname}.json').read_text())['groups']
    return {im['name']: im['psnr'] for v in g.values() for im in v['images']}
dc = {r: per_image(r, 'docaligner') for r in main.index if (RES / f'{r}__docaligner.json').exists()}
bad = ['26_1.png', '26_2.png', '8_2.png']
t2 = pd.DataFrame({r: {'3 ảnh nền màu (PSNR dB)': np.mean([v[k] for k in bad])} for r, v in dc.items()}).T
print('Trang nền màu của DocAligner (26_1, 26_2, 8_2)'); display(t2.round(2))
t2.round(3).to_csv(RES / 'mau_nen.csv'); ci.round(3).to_csv(RES / 'anh_sach.csv')


## 6. CER (VietOCR gốc, 317 dòng ESP32)


In [ ]:
# vietocr==0.3.13 ghim pillow==10.2.0 (không có wheel cho Python 3.13): cài --no-deps rồi các gói cần dùng
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'vietocr==0.3.13'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'gdown', 'prefetch_generator', 'lmdb', 'rapidfuzz', 'pypdfium2', 'pyyaml', 'requests', 'tqdm'], check=True)
WEIGHTS = '/content/vgg_transformer.pth'
shutil.copy(D / 'data' / 'vgg_transformer.pth', WEIGHTS)     # VietOCR gốc, chưa fine-tune
cer_rows = []
variants = ['esp_raw', 'scan'] + ['esp_' + r for r in CKPTS]
for v in variants:
    r = subprocess.run([sys.executable, C / 'n2_eval.py', '--data', E, '--skip-labels', '--variants', v, '--split', 'test',
                        '--device', 'cuda', '--weights', WEIGHTS, '--tag', 'retrain3'], capture_output=True, text=True)
    line = [l for l in r.stdout.splitlines() if l.startswith(v + ' ')]
    if line:
        o = json.loads(line[0][len(v) + 1:])['all']
        cer_rows.append({'phiên bản': v.replace('esp_', ''), 'CER': o['cer'], 'CER bỏ dấu': o['cer_no_accent'], 'lỗi riêng do dấu': round(o['cer'] - o['cer_no_accent'], 4)})
    else:
        print(v, 'lỗi:', r.stderr[-300:])
cer = pd.DataFrame(cer_rows).set_index('phiên bản')
display(cer.style.format('{:.3f}').highlight_min(axis=0, color='#c6efce'))
cer.to_csv(RES / 'cer_esp32.csv')
ax = cer[['CER', 'CER bỏ dấu']].plot.bar(figsize=(9, 4), rot=30); ax.set_ylabel('CER'); ax.set_title('CER trên 317 dòng ESP32 test (thấp hơn là tốt hơn)')
plt.tight_layout(); plt.savefig(FIG / 'cer_esp32.png', dpi=110); plt.show()


In [ ]:
try:
    from google.colab import _message
    nb = _message.blocking_request('get_ipynb', request='', timeout_sec=120)['ipynb']
    (RES / 'PBL6_Colab_Seeds_da_chay.ipynb').write_text(json.dumps(nb, ensure_ascii=False), encoding='utf-8')
except Exception as e:
    print('Không tự lưu được, dùng File → Download .ipynb:', e)
print(sorted(p.name for p in RES.iterdir()))
